# Unit 5 Plotly Visualisation Exercise

This task experiments with the data visualisation library, Plotly, commonly used to visualise machine learning (ML) models. The library is used to create interactive graphs and plots. 

This exercise is to explore the effect of changing the types of chart, as well as colours and variables, and understand its impact on data visualisation. 

In [ ]:
%pip install plotly
%pip install statsmodels
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Ordinary Least Squares (OLS) with plotly.express

From the original code, I changed the opacity and spliced out the colour by 'sex', which produced a separate OLS trendline for each group. 

The relationship between bill and tip is similar across sexes. 

In [4]:
# Original code

df = px.data.tips()
fig = px.scatter(
    df, x='total_bill', y='tip', opacity=0.65,
    trendline='ols', trendline_color_override='darkblue'
)
fig.show()

# Altered code with opacity and colour specified

df = px.data.tips()
fig = px.scatter(
    df, x='total_bill', y='tip', opacity=0.4,
    color='sex',
    trendline='ols', trendline_color_override='red'
)
fig.show()

### Linear Regression with scikit-learn

From the original code, I changed the line and scatter colour, style of dash, and the number of prediction points. Changing teh regression line to a dashed one makes it distinct from the data. Colouring by 'time' shows whether tips differ during lunch vs dinner. 

In [5]:
# Original code

df = px.data.tips()
X = df.total_bill.values.reshape(-1, 1)

model = LinearRegression()
model.fit(X, df.tip)

x_range = np.linspace(X.min(), X.max(), 100)
y_range = model.predict(x_range.reshape(-1, 1))

fig = px.scatter(df, x='total_bill', y='tip', opacity=0.65)
fig.add_traces(go.Scatter(x=x_range, y=y_range, name='Regression Fit'))
fig.show()

# Altered code with line & scatter colour, dash style, and no. prediction points changed

df = px.data.tips()
X = df.total_bill.values.reshape(-1, 1)

model = LinearRegression()
model.fit(X, df.tip)

x_range = np.linspace(X.min(), X.max(), 10)  # changed: 100 -> 10
y_range = model.predict(x_range.reshape(-1, 1))

fig = px.scatter(df, x='total_bill', y='tip', opacity=0.65, color='time')  # changed: added color
fig.add_traces(go.Scatter(
    x=x_range, y=y_range, name='Regression Fit',
    line=dict(color='red', dash='dash')  # changed: colour and dash
))
fig.show()

### Model Generalisation on Unseen Data

I expeeimented with changing the test size to 0.5 to train the model on half the dataset, and changed random_state to produce a different split each time. This shows how results can vary by change. 

In [10]:
# Original code

df = px.data.tips()
X = df.total_bill.to_numpy()[:, None]
X_train, X_test, y_train, y_test = train_test_split(X, df.tip, random_state=0)

model = LinearRegression()
model.fit(X_train, y_train)

x_range = np.linspace(X.min(), X.max(), 100)
y_range = model.predict(x_range.reshape(-1, 1))

fig = go.Figure([
    go.Scatter(x=X_train.squeeze(), y=y_train, name='train', mode='markers'),
    go.Scatter(x=X_test.squeeze(), y=y_test, name='test', mode='markers'),
    go.Scatter(x=x_range, y=y_range, name='prediction')
])
fig.show()

# Altered code with different test size, random_state, marker colours and sizes

df = px.data.tips()
X = df.total_bill.to_numpy()[:, None]
X_train, X_test, y_train, y_test = train_test_split(
    X, df.tip, random_state=42, test_size=0.5  # changed: random_state and test_size
)

model = LinearRegression()
model.fit(X_train, y_train)

x_range = np.linspace(X.min(), X.max(), 100)
y_range = model.predict(x_range.reshape(-1, 1))

fig = go.Figure([
    go.Scatter(x=X_train.squeeze(), y=y_train, name='train', mode='markers',
               marker=dict(color='royalblue', size=9)),   # changed: colour and size
    go.Scatter(x=X_test.squeeze(), y=y_test, name='test', mode='markers',
               marker=dict(color='tomato', size=9)),      # changed: colour and size
    go.Scatter(x=x_range, y=y_range, name='prediction',
               line=dict(color='green', width=3))          # changed: line colour
])
fig.show()

### Comparing Different kNN Model Parameters

k was changed from 10 to 3, and a third model with k = 50 was added. The line with k = 3 became wiggled and followed individual points, a sign of overfitting. The line with k = 50 became overly smooth, a sign of underfitting. 

In [12]:
# Original code

df = px.data.tips()
X = df.total_bill.values.reshape(-1, 1)
x_range = np.linspace(X.min(), X.max(), 100)

    # Model #1
knn_dist = KNeighborsRegressor(10, weights='distance')
knn_dist.fit(X, df.tip)
y_dist = knn_dist.predict(x_range.reshape(-1, 1))

    # Model #2
knn_uni = KNeighborsRegressor(10, weights='uniform')
knn_uni.fit(X, df.tip)
y_uni = knn_uni.predict(x_range.reshape(-1, 1))

fig = px.scatter(df, x='total_bill', y='tip', color='sex', opacity=0.65)
fig.add_traces(go.Scatter(x=x_range, y=y_uni, name='Weights: Uniform'))
fig.add_traces(go.Scatter(x=x_range, y=y_dist, name='Weights: Distance'))
fig.show()

# Altered code

df = px.data.tips()
X = df.total_bill.values.reshape(-1, 1)
x_range = np.linspace(X.min(), X.max(), 100)

knn_3 = KNeighborsRegressor(3, weights='uniform')   # changed: k=3 (overfit)
knn_3.fit(X, df.tip)
y_3 = knn_3.predict(x_range.reshape(-1, 1))

knn_10 = KNeighborsRegressor(10, weights='uniform')
knn_10.fit(X, df.tip)
y_10 = knn_10.predict(x_range.reshape(-1, 1))

knn_50 = KNeighborsRegressor(50, weights='uniform')  # changed: k=50 (underfit)
knn_50.fit(X, df.tip)
y_50 = knn_50.predict(x_range.reshape(-1, 1))

fig = px.scatter(df, x='total_bill', y='tip', color='smoker', opacity=0.5)  # changed: color='smoker'
fig.add_traces(go.Scatter(x=x_range, y=y_3,  name='k=3  (overfit)',  line=dict(color='red')))
fig.add_traces(go.Scatter(x=x_range, y=y_10, name='k=10 (balanced)', line=dict(color='green')))
fig.add_traces(go.Scatter(x=x_range, y=y_50, name='k=50 (underfit)', line=dict(color='purple')))
fig.show()


### Visualising Coefficients for Multiple Linear Regression (MLR)

Changing the target from petal width to sepal width changes which features are the most influenctial. Some coeefficients change, meanign features which previously increased predictions now decrease them. This shows how the same features can have very different relationships with different targets. 

In [3]:
# Original code

df = px.data.iris()

X = df.drop(columns=['petal_width', 'species_id'])
X = pd.get_dummies(X, columns=['species'], prefix_sep='=')
y = df['petal_width']

model = LinearRegression()
model.fit(X, y)

colors = ['Positive' if c > 0 else 'Negative' for c in model.coef_]

fig = px.bar(
    x=X.columns, y=model.coef_, color=colors,
    color_discrete_sequence=['red', 'blue'],
    labels=dict(x='Feature', y='Linear coefficient'),
    title='Weight of each feature for predicting petal width'
)
fig.show()

# Altered code

df = px.data.iris()

X = df.drop(columns=['sepal_length', 'species_id'])    # changed: new target
X = pd.get_dummies(X, columns=['species'], prefix_sep='=')
y = df['sepal_length']                                  # changed: target variable

model = LinearRegression()
model.fit(X, y)

colors = ['Positive' if c > 0 else 'Negative' for c in model.coef_]

fig = px.bar(
    x=X.columns, y=model.coef_, color=colors,
    color_discrete_sequence=['darkorange', 'steelblue'],  # changed: colours
    labels=dict(x='Feature', y='Linear coefficient'),
    title='Weight of each feature for predicting sepal length'  # changed: title
)
fig.show()

### Prediction Error - Simple Actual vs Predicted

Adding all four features moves points closer to the diagonal, improving prediction accuracy. Colouring by species makes it clear that prediction errors are not random and that particular species are over or under-predicted, indicating that species in an important compounding variable. 

In [4]:
# Original code

df = px.data.iris()
X = df[['sepal_width', 'sepal_length']]
y = df['petal_width']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

fig = px.scatter(x=y, y=y_pred, labels={'x': 'ground truth', 'y': 'prediction'})
fig.add_shape(
    type="line", line=dict(dash='dash'),
    x0=y.min(), y0=y.min(),
    x1=y.max(), y1=y.max()
)
fig.show()

# Altered code

df = px.data.iris()
X = df[['sepal_width', 'sepal_length', 'petal_length']]  # changed: added petal_length
y = df['petal_width']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

fig = px.scatter(
    x=y, y=y_pred,
    color=df['species'],                                  # changed: colour by species
    labels={'x': 'ground truth', 'y': 'prediction', 'color': 'species'}
)
fig.add_shape(
    type="line", line=dict(dash='dash', color='black'),  # changed: line colour
    x0=y.min(), y0=y.min(),
    x1=y.max(), y1=y.max()
)
fig.show()

### Enhanced Prediction Error Analysis using plotly.express

Using box plots summarises the spread and median of actual versus predicted values, more concisely than histograms do. A larger test set produces a more reliable estimate of generalisations but leaves less data for training which can increase the gap between train and test trends. 

In [5]:
# Original code

df = px.data.iris()

train_idx, test_idx = train_test_split(df.index, test_size=.25, random_state=0)
df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'

X = df[['sepal_width', 'sepal_length']]
y = df['petal_width']
X_train = df.loc[train_idx, ['sepal_width', 'sepal_length']]
y_train = df.loc[train_idx, 'petal_width']

model = LinearRegression()
model.fit(X_train, y_train)
df['prediction'] = model.predict(X)

fig = px.scatter(
    df, x='petal_width', y='prediction',
    marginal_x='histogram', marginal_y='histogram',
    color='split', trendline='ols'
)
fig.update_traces(histnorm='probability', selector={'type':'histogram'})
fig.add_shape(
    type="line", line=dict(dash='dash'),
    x0=y.min(), y0=y.min(),
    x1=y.max(), y1=y.max()
)
fig.show()

# Altered code

df = px.data.iris()

train_idx, test_idx = train_test_split(df.index, test_size=.5, random_state=0)  # changed: test_size
df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'

X = df[['sepal_width', 'sepal_length']]
y = df['petal_width']
X_train = df.loc[train_idx, ['sepal_width', 'sepal_length']]
y_train = df.loc[train_idx, 'petal_width']

model = LinearRegression()
model.fit(X_train, y_train)
df['prediction'] = model.predict(X)

fig = px.scatter(
    df, x='petal_width', y='prediction',
    marginal_x='box', marginal_y='box',   # changed: histogram -> box
    color='split', trendline='ols'
)
fig.add_shape(
    type="line", line=dict(dash='dash'),
    x0=y.min(), y0=y.min(),
    x1=y.max(), y1=y.max()
)
fig.show()